# 06 · Exportar los modelos a ONNX para la página

La página publicada en Hugging Face es un Space estático (gratis, sin servidor). Para que el modelo corra igual en vivo, los 12 modelos de `datos_app/modelos` (6 campañas × enero/completa) se convierten a **ONNX** y la página los ejecuta en el navegador con *onnxruntime-web* (`static/modelo.js`).

El notebook verifica que cada modelo ONNX dé lo mismo que el de Keras, con la misma tolerancia que los tests de la API: diferencia máxima de probabilidad menor a 0,01 y la misma clase en al menos el 99,5 % de los lotes. El resultado queda en `datos_app/onnx/` y se sube tal cual a la carpeta `onnx/` del Space.

Corre en CPU (no hace falta GPU) y tarda unos minutos.

In [ ]:
!pip install -q tf2onnx onnx onnxruntime

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    APP_DRIVE = '/content/drive/MyDrive/TP_final_IA2/datos_app'
except ImportError:
    APP_DRIVE = os.path.abspath('datos_app')
print(APP_DRIVE, os.listdir(APP_DRIVE))

In [ ]:
import sys
sys.argv = ['convertir']   # usa APP_DRIVE
import os, sys, json, glob, numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf, keras, tf2onnx, onnxruntime as ort
sys.path.insert(0, sys.argv[1] if len(sys.argv) > 1 else APP_DRIVE); from inferencia import preprocesar, CLASES
APP = sys.argv[1] if len(sys.argv) > 1 else APP_DRIVE; SAL = os.path.join(APP, 'onnx'); os.makedirs(SAL, exist_ok=True)
indice = json.load(open(f'{APP}/modelos/indice.json'))

def convertir(ruta_keras, T, ruta_onnx):
    m = keras.models.load_model(ruta_keras, compile=False)
    spec = (tf.TensorSpec((None, T, 16, 16, 12), tf.float32, name='serie'),)
    @tf.function(input_signature=spec)
    def f(serie):
        out = m({'serie': serie} if isinstance(m.input, dict) else serie, training=False)
        if not isinstance(out, dict):
            out = dict(zip(m.output_names, out))
        return {'cultivo': tf.cast(out['cultivo'], tf.float32), 'rinde': tf.cast(out['rinde'], tf.float32)}
    tf2onnx.convert.from_function(f, input_signature=spec, opset=17, output_path=ruta_onnx)
    return m

if os.path.exists(f'{APP}/ejemplos/series.npz'):
    bandas = np.load(f'{APP}/ejemplos/series.npz', allow_pickle=True)['bandas']
else:
    bandas = np.stack([np.load(f)['bandas'] for f in sorted(glob.glob(f'{APP}/ejemplos/descargas/*.npz'))])
print('lotes para verificar:', len(bandas))
x12 = preprocesar(bandas)
nuevo, peor = {}, 0
for camp, modos in indice.items():
    nuevo[camp] = {}
    for modo, info in modos.items():
        T = info['periodos']; arch = info['archivo'].replace('.keras', '.onnx')
        m = convertir(f"{APP}/modelos/{info['archivo']}", T, f'{SAL}/{arch}')
        mu = np.asarray(info['mu_canales'], np.float32); sd = np.asarray(info['sd_canales'], np.float32)
        x = ((x12[:, :T] - mu) / sd).astype(np.float32)
        ok = m.predict({'serie': x} if isinstance(m.input, dict) else x, verbose=0)
        s = ort.InferenceSession(f'{SAL}/{arch}', providers=['CPUExecutionProvider'])
        po, ro = s.run(['cultivo', 'rinde'], {'serie': x})
        dif = float(np.abs(po - ok['cultivo']).max()); igual = float((po.argmax(1) == ok['cultivo'].argmax(1)).mean())
        drinde = float(np.abs(ro.ravel() - ok['rinde'].ravel()).max() * info['rinde_desvio'])
        print(f'{camp} {modo}: dif. máx. prob {dif:.2e} · misma clase {100*igual:.1f} % · dif. máx. rinde {drinde:.2f} kg/ha · {os.path.getsize(f"{SAL}/{arch}")/1e6:.2f} MB')
        assert dif < 0.01 and igual >= 0.995, 'el ONNX no reproduce al modelo de Keras'
        nuevo[camp][modo] = {k: v for k, v in info.items() if k != 'archivo'} | {'archivo': arch}
json.dump({'clases': CLASES, 'bandas': ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12'], 'modelos': nuevo},
          open(f'{SAL}/indice.json', 'w'), ensure_ascii=False)
print('listo:', sorted(os.listdir(SAL)))
